# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring
**Builds on:** `w01_research_question.ipynb` (decision, action, cost of a wrong call) and
`w02_ml_task_framing.ipynb` (task type, proxy target, Precision@50 named before training).

> Skills loaded for this assignment: `skills/writing-data-contracts/SKILL.md` +
> `skills/flyrank/flyrank-data/SKILL.md`.

**What changes in this notebook.** W01 and W02 ran on the 30k-row starter slice, where one row
was one page over a *trailing* 90-day window and the label was `trend_direction == "down"` —
a rule computed on the same window the features came from. W02 named that as the weak point and
said the cleaner target would be *an outcome measured in a later time window*. This notebook is
where that promise gets kept: I move to the warehouse, choose an explicit decision date, and
define the label on a **forward** window that no feature is allowed to see.

**How to run this.** Colab. Request access to
`FlyRank/internship-warehouse` on Hugging Face, create a **plain Read** token, and store it as a
Colab Secret named `HF_TOKEN` (key panel, left sidebar). Never paste the token into a cell —
this repo is public.

**One thing I deliberately did not do:** the `_sample` table is exactly the final month
(June 2026), which is the natural outcome window of any past→future label. I develop on
`month=2026-03` and treat the final month as a sealed test month. It is not read anywhere below.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The contract, in five plain answers

**1) What one row means.**
One row = **one content item (`content_hash_id`) evaluated at one decision date, 2026-03-31.**

Not one row per day. The warehouse fact table's grain is `report_date × client × content`; my
*decision* grain is coarser, because the decision a reviewer makes is "open this page or not",
once, not "open this page on the 14th of March". So the daily fact table is the **input** grain
and the feature frame is the **decision** grain, and section 3 verifies both separately.

**2) Which tables.**
`fact_content_daily_performance` only, partitions `month=2026-03` (features) and
`month=2026-04` (label). No dimension join is needed for these five features; `dim_clients`
would be needed the moment I widen the window, because `gsc_data_start` differs per client.

**3) Which time window.**

```text
feature window   2026-03-01 .. 2026-03-31   (31 days, everything the model may see)
decision date    2026-03-31                 (the moment a reviewer would act)
label window     2026-04-01 .. 2026-04-30   (30 days, the future — no feature may touch it)
```

The two windows do not overlap by a single day. That gap is the whole point of the contract.

**4) What I predict / rank.**
`will_decline_next_30d` = 1 when a page's April impressions fall below 80% of its March
impressions. It mirrors FlyRank's −20% convention so my numbers stay comparable to the starter
pipeline, but it is a **forward-looking observed outcome**, not a rule read off the same window.
Consistent with W02, the model's probability is used as a **ranking score**, and the metric is
**Precision@50** reported beside the base rate.

**5) One thing I deliberately exclude.**
`fact_content_query_90d`. Its fixed 90-day window covers the most recent ~3 months of the
snapshot, which **contains my April label window**. Any `impressions_90d` or `*_last30` column
from it is future information wearing a feature's clothes. I exclude the whole table rather than
cherry-pick safe columns from it — a smaller contract I can defend beats a wider one I can't.

*(Secondary exclusion, same section 2 table: AI-referral columns. 30,177 rows carry AI sessions
against 78.8M daily rows. Too sparse to be a feature at this grain; noted as a limitation, not
used.)*

In [1]:
# --- Setup: install, authenticate, discover the release ----------------------
# Nothing below is typed from memory: file paths and column names are DISCOVERED
# and printed, so if the release layout differs from my assumptions the notebook
# says so instead of silently computing the wrong thing.

!pip -q install --upgrade duckdb huggingface_hub scikit-learn pandas

import os, sys

TOKEN = None
try:                                    # Colab
    from google.colab import userdata
    TOKEN = userdata.get("HF_TOKEN")
except Exception:
    TOKEN = os.environ.get("HF_TOKEN")

assert TOKEN, (
    "No HF_TOKEN found. In Colab: key icon -> add a Secret named HF_TOKEN "
    "(plain Read token, gated-repo access ticked). Never paste it into a cell."
)
print("HF token loaded from the secret store (value never printed).")

AssertionError: No HF_TOKEN found. In Colab: key icon -> add a Secret named HF_TOKEN (plain Read token, gated-repo access ticked). Never paste it into a cell.

In [ ]:
from huggingface_hub import list_repo_files

REPO      = "FlyRank/internship-warehouse"
FACT      = "fact_content_daily_performance"
FEAT_MONTH  = "2026-03"     # mid-panel month, per the assignment warning
LABEL_MONTH = "2026-04"     # the forward window

files = list_repo_files(REPO, repo_type="dataset", token=TOKEN)
print(f"Files visible in the release: {len(files):,}")

# What top-level objects exist? (printed, not assumed)
tops = sorted({f.split("/")[0] for f in files})
print("\nTop-level objects:")
for t in tops:
    n = sum(1 for f in files if f.startswith(t))
    print(f"   {t:<45} {n:>6,} files")

def partition_files(month):
    hits = [f for f in files
            if FACT in f and f.endswith(".parquet")
            and f"month={month}" in f and "_sample" not in f]
    return sorted(hits)

feat_files  = partition_files(FEAT_MONTH)
label_files = partition_files(LABEL_MONTH)

print(f"\nmonth={FEAT_MONTH}  -> {len(feat_files):,} parquet parts")
print(f"month={LABEL_MONTH}  -> {len(label_files):,} parquet parts")
assert feat_files and label_files, (
    "Partition layout is not what I expected. Inspect `files` above and adjust "
    "partition_files() before going further — do NOT guess."
)
print("\nExample part:", feat_files[0])

In [ ]:
from huggingface_hub import hf_hub_download

def fetch(paths):
    return [hf_hub_download(REPO, p, repo_type="dataset", token=TOKEN) for p in paths]

local_feat  = fetch(feat_files)
local_label = fetch(label_files)
print(f"Cached locally: {len(local_feat)} feature parts + {len(local_label)} label parts")

In [ ]:
import duckdb

con = duckdb.connect()

def as_sql_list(paths):
    return "[" + ",".join("'" + p + "'" for p in paths) + "]"

con.execute(f"""
    CREATE OR REPLACE VIEW daily AS
    SELECT * FROM read_parquet({as_sql_list(local_feat + local_label)}, union_by_name = true)
""")

schema = con.execute("DESCRIBE daily").df()
print("Columns actually present in the daily fact table:\n")
print(schema.to_string(index=False))

In [ ]:
# --- Resolve logical names -> real column names ------------------------------
# The data dictionary names some columns differently from the starter CSV, so I
# resolve them against the printed schema instead of hard-coding and hoping.

present = list(schema["column_name"])

def pick(*candidates, required=True, label=""):
    for c in candidates:
        if c in present:
            return c
    if required:
        raise KeyError(f"None of {candidates} found for '{label}'. Schema: {present}")
    return None

COL = {
    "date":    pick("report_date", "date", label="date"),
    "client":  pick("client_hash_id", "client_id", label="client"),
    "content": pick("content_hash_id", "content_id", label="content"),
    "impr":    pick("impressions", "gsc_impressions", "impressions_total", label="impressions"),
    "clicks":  pick("clicks", "gsc_clicks", label="clicks"),
    "ga4":     pick("ga4_data_available", required=False, label="ga4 flag"),
}
# position may ship as a daily average or as a GSC-style sum; both are handled
COL["avg_pos"] = pick("avg_position", "position", required=False, label="avg position")
COL["sum_pos"] = pick("sum_position", "sum_top_position", required=False, label="sum position")

for k, v in COL.items():
    print(f"   {k:<9} -> {v}")

if COL["avg_pos"]:
    POS_SQL = (f"SUM({COL['impr']} * {COL['avg_pos']}) "
               f"/ NULLIF(SUM(CASE WHEN {COL['avg_pos']} > 0 THEN {COL['impr']} END), 0)")
    print("\nPosition: impressions-weighted mean of the daily average.")
elif COL["sum_pos"]:
    POS_SQL = f"SUM({COL['sum_pos']}) / NULLIF(SUM({COL['impr']}), 0) + 1"
    print("\nPosition: GSC convention -> SUM(sum_position)/SUM(impressions) + 1.")
else:
    POS_SQL = None
    print("\nNo position column in this release — the feature frame will carry four features, "
          "not five, and section 3 says so explicitly.")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `content_hash_id` | context | Decision key and dedup key. Never a feature — it is a pseudonym, and an id as a feature is memorisation. |
| `client_hash_id` | context | Grouping key for the held-out split. Never a feature: with 70 clients the model would learn "client 12 declines" instead of learning content. |
| `report_date` | context | Window filter. Defines which side of the wall a row sits on. |
| impressions (March) | **feature** | Demand size at the decision moment. |
| clicks (March) | **feature** (via CTR) | Capture rate at the decision moment. |
| position (March) | **feature** | Where the page sat in results during March. |
| impressions per day (March) | **feature** (via active days + momentum) | Consistency and within-window direction. |
| impressions (April) | **label only** | The outcome. Touching it as a feature is the trap in section 3 — and I run it on purpose there, then delete it. |
| `ga4_data_available` | context | Availability flag. Checked with `IS TRUE` in Q3, used to explain zeros — not fed to the model. |
| GA4 sessions / engagement / scroll | excluded | Systematically missing before each client's `ga4_data_start`, and the missingness follows client identity. Imputing zeros would encode "which client" into the features. Revisit once the panel is filtered per client. |
| AI-referral columns | excluded | 30,177 rows carry AI sessions against 78.8M daily rows. At this grain the column is almost constant; a feature that is almost always zero teaches nothing and invites over-reading. |
| `fact_content_query_90d` (whole table) | excluded | Its fixed 90-day window overlaps my April label window. Every aggregate in it is partly made of the answer. |
| `health_score`, `priority_score`, `action_type` | excluded | Not shipped in the release by design. If I ever rebuilt one, it would be a baseline to beat, never a feature — that is the circular-result trap. |

In [ ]:
# --- The buckets, as code, so they can be enforced rather than promised ------
CONTRACT = {
    "context":  [COL["content"], COL["client"], COL["date"]] + ([COL["ga4"]] if COL["ga4"] else []),
    "feature_sources": [COL["impr"], COL["clicks"]] + [c for c in (COL["avg_pos"], COL["sum_pos"]) if c],
    "label_source":    [COL["impr"]],   # same column, DIFFERENT window - that is the whole discipline
    "excluded_prefixes": ["session", "engag", "scroll", "pageview", "user", "ai_", "_ai",
                          "health", "priority", "action", "refresh_tier"],
}

excluded_here = [c for c in present
                 if any(p in c.lower() for p in CONTRACT["excluded_prefixes"])]

print("Context columns :", CONTRACT["context"])
print("Feature sources :", CONTRACT["feature_sources"])
print("Label source    :", CONTRACT["label_source"], "(April window only)")
print(f"\nExcluded by rule ({len(excluded_here)} columns present in this release):")
for c in excluded_here:
    print("   -", c)
if not excluded_here:
    print("   (none of the excluded families ship in this table)")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims, three queries:

1. **Grain** — one input row really is `date × client × content`, and one feature-frame row
   really is one content item.
2. **Size and span** — how many rows my slice has, and that its dates stop exactly where the
   contract says they stop.
3. **Availability** — how many rows survive `ga4_data_available IS TRUE`.

In [ ]:
# === Q1 — GRAIN =============================================================
q1 = con.execute(f"""
    SELECT
        COUNT(*)                                                       AS rows_total,
        COUNT(DISTINCT CONCAT_WS('|', CAST({COL['date']} AS VARCHAR),
                                      {COL['client']}, {COL['content']})) AS distinct_keys,
        COUNT(DISTINCT {COL['content']})                               AS distinct_content,
        COUNT(DISTINCT {COL['client']})                                AS distinct_clients
    FROM daily
    WHERE {COL['date']} BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

print("Q1 - is one row really date x client x content, in month=2026-03?\n")
print(q1.to_string(index=False))

dup = int(q1.rows_total[0] - q1.distinct_keys[0])
print(f"\nduplicate keys: {dup:,}")
print("-> grain CONFIRMED." if dup == 0 else
      "-> grain VIOLATED. Stop and investigate before aggregating anything.")

In [ ]:
# === Q2 — SIZE AND DATE SPAN ================================================
q2 = con.execute(f"""
    SELECT
        CASE WHEN {COL['date']} <= DATE '2026-03-31' THEN 'features (March)'
             ELSE 'label (April)' END                AS window,
        COUNT(*)                                     AS rows,
        MIN({COL['date']})                           AS first_date,
        MAX({COL['date']})                           AS last_date,
        COUNT(DISTINCT {COL['date']})                AS n_days,
        COUNT(DISTINCT {COL['content']})             AS content_items,
        COUNT(DISTINCT {COL['client']})              AS clients
    FROM daily
    GROUP BY 1
    ORDER BY 1
""").df()

print("Q2 - my slice, by window:\n")
print(q2.to_string(index=False))
print("\nThe two windows must not touch: last feature date < first label date.")

In [ ]:
# === Q3 — AVAILABILITY (IS TRUE, not = TRUE) ================================
# `= TRUE` drops NULLs silently and returns NULL for them; `IS TRUE` is NULL-safe
# and answers the question actually being asked: how many rows are *known* to
# have GA4 tracking on?

if COL["ga4"]:
    q3 = con.execute(f"""
        SELECT
            COUNT(*)                                                  AS rows_total,
            COUNT(*) FILTER (WHERE {COL['ga4']} IS TRUE)               AS ga4_true,
            COUNT(*) FILTER (WHERE {COL['ga4']} IS FALSE)              AS ga4_false,
            COUNT(*) FILTER (WHERE {COL['ga4']} IS NULL)               AS ga4_null,
            ROUND(100.0 * COUNT(*) FILTER (WHERE {COL['ga4']} IS TRUE)
                  / NULLIF(COUNT(*), 0), 2)                            AS pct_true
        FROM daily
        WHERE {COL['date']} BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    """).df()
    print("Q3 - GA4 availability in month=2026-03:\n")
    print(q3.to_string(index=False))
    print("\nRows that are NOT TRUE carry zero-filled GA4 columns. Those zeros mean")
    print("'tracking was off', not 'no engagement' - which is exactly why every GA4")
    print("column sits in the excluded bucket for this pass.")
else:
    print("No ga4_data_available column in this release; nothing to filter with IS TRUE.")

### Five features, and when each one is knowable

Every feature below is computed from **March only**. The decision date is 2026-03-31, so the
test each one has to pass is: *would a reviewer standing on 31 March already have this number?*

| Feature | Definition | Knowable at the decision moment because… |
|---|---|---|
| `log_impressions_march` | `log1p(SUM(impressions))`, 1–31 March | …it is a sum over days that have already happened. `log1p` because traffic is heavy-tailed and one 30,000-impression page would otherwise dominate the scale. |
| `ctr_march` | `100 * clicks / impressions`, March | …both inputs are March measurements. Kept on the ×100 convention the data dictionary uses, so `0.76` reads as 0.76%. |
| `avg_position_march` | impressions-weighted mean position, March | …it describes where the page already sat during March. Weighted, not a plain mean, so a day with 2 impressions doesn't count as much as a day with 900. |
| `active_days_march` | count of March days with ≥1 impression (0–31) | …it is a property of the elapsed window. It separates "steady 100 impressions across 31 days" from "one spike and silence" — two very different pages with identical totals. |
| `momentum_within_march` | `(impr 22–31 Mar + 1) / (impr 1–10 Mar + 1)` | …both halves sit **inside** the feature window. This is the safe version of a trend: it looks at direction without crossing into April. The `+1` keeps it finite when a page had no impressions early in the month. |

Deliberately absent: anything dated April, anything from the 90-day query table, anything from
GA4. The label is `will_decline_next_30d`, built from April impressions and nothing else.

In [ ]:
# --- Build the feature frame at the DECISION grain ---------------------------
pos_select = f"{POS_SQL} AS avg_position_march," if POS_SQL else "NULL AS avg_position_march,"

frame = con.execute(f"""
    WITH march AS (
        SELECT
            {COL['content']}                          AS content_id,
            ANY_VALUE({COL['client']})                AS client_id,
            SUM({COL['impr']})                        AS impressions_march,
            SUM({COL['clicks']})                      AS clicks_march,
            {pos_select}
            COUNT(DISTINCT CASE WHEN {COL['impr']} > 0 THEN {COL['date']} END)
                                                      AS active_days_march,
            SUM(CASE WHEN {COL['date']} >= DATE '2026-03-22' THEN {COL['impr']} ELSE 0 END)
                                                      AS impr_late_march,
            SUM(CASE WHEN {COL['date']} <= DATE '2026-03-10' THEN {COL['impr']} ELSE 0 END)
                                                      AS impr_early_march,
            MIN({COL['date']})                        AS first_seen_march
        FROM daily
        WHERE {COL['date']} BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
        GROUP BY 1
    ),
    april AS (
        SELECT {COL['content']} AS content_id, SUM({COL['impr']}) AS impressions_april
        FROM daily
        WHERE {COL['date']} BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
        GROUP BY 1
    )
    SELECT
        m.*,
        COALESCE(a.impressions_april, 0) AS impressions_april,
        CASE WHEN COALESCE(a.impressions_april, 0) < 0.8 * m.impressions_march
             THEN 1 ELSE 0 END           AS will_decline_next_30d
    FROM march m
    LEFT JOIN april a USING (content_id)
    WHERE m.impressions_march >= 100          -- eligibility: measurable demand
""").df()

import numpy as np

frame["log_impressions_march"] = np.log1p(frame["impressions_march"])
frame["ctr_march"]             = 100 * frame["clicks_march"] / frame["impressions_march"]
frame["momentum_within_march"] = ((frame["impr_late_march"] + 1)
                                  / (frame["impr_early_march"] + 1))

FEATURES = ["log_impressions_march", "ctr_march", "active_days_march", "momentum_within_march"]
if POS_SQL:
    frame["avg_position_march"] = frame["avg_position_march"].fillna(0)
    FEATURES.insert(2, "avg_position_march")

print(f"Feature frame: {len(frame):,} rows x {len(FEATURES)} features")
print(f"Features: {FEATURES}")

# grain check at the DECISION grain, not the input grain
print(f"\nrows: {len(frame):,}   unique content_id: {frame.content_id.nunique():,}")
print("-> decision grain CONFIRMED (one row = one content item)."
      if len(frame) == frame.content_id.nunique() else "-> DUPLICATES. Stop.")

base_rate = frame["will_decline_next_30d"].mean()
print(f"\nBase rate (share declining in April): {base_rate:.3f}  ({base_rate:.1%})")
print(f"-> a random draw of 50 pages scores Precision@50 ~= {base_rate:.3f}")
print("   Every precision number below is read against THIS, per W01/W02.")
print(frame[FEATURES].describe().T.to_string())

### The trap, run on purpose

W02 argued about leakage in prose. Here it gets performed. I add **one** column that is derived
from the label window — April impressions, the exact input the label is computed from — score the
model, watch the number go where no honest model goes, then delete the column and keep the
number I can defend.

The point is not that the score rises. It is *how far* it rises, and how ordinary the leaking
column looks sitting in a dataframe next to four legitimate ones.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k=50):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean()), k

assert frame["will_decline_next_30d"].nunique() == 2, (
    "The label has only one class. That is a contract problem, not a modelling one: "
    "either the eligibility filter is too tight or the -20% rule doesn't bite on this "
    "month. Fix section 1 before touching a model."
)

def quick_score(features, tag):
    X, y, g = frame[features].values, frame["will_decline_next_30d"].values, frame["client_id"].values
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, g))
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        print(f"{tag:<28} SKIPPED - the client-held-out split put one class entirely on "
              "one side. Try another random_state, or report per-client base rates first.")
        return float("nan"), float("nan")
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(X[tr], y[tr])
    p = model.predict_proba(X[te])[:, 1]
    auc = roc_auc_score(y[te], p)
    p50, k_used = precision_at_k(y[te], p, 50)
    note = "" if k_used == 50 else f" [K={k_used}, test set smaller than 50]"
    print(f"{tag:<28} ROC-AUC {auc:.3f}   Precision@50 {p50:.3f}   "
          f"(test base rate {y[te].mean():.3f}, clients held out){note}")
    return auc, p50

print("Client-held-out split, logistic regression, no tuning.\n")
auc_honest, p50_honest = quick_score(FEATURES, "1. contract features only")

# --- deliberate leak --------------------------------------------------------
frame["LEAK_impressions_april"] = np.log1p(frame["impressions_april"])
auc_leak, p50_leak = quick_score(FEATURES + ["LEAK_impressions_april"], "2. + ONE label-window column")

print(f"\nDelta from one leaking column: AUC {auc_leak - auc_honest:+.3f}, "
      f"Precision@50 {p50_leak - p50_honest:+.3f}")

In [ ]:
# --- delete it and keep the honest number ------------------------------------
frame = frame.drop(columns=["LEAK_impressions_april"])
assert not any("april" in c.lower() for c in FEATURES), "A label-window column is still in FEATURES."
assert "LEAK_impressions_april" not in frame.columns

auc_final, p50_final = quick_score(FEATURES, "3. after deleting the leak")

print(f"""
The number I keep: ROC-AUC {auc_final:.3f}, Precision@50 {p50_final:.3f},
against a base rate of {base_rate:.3f}.

What the experiment showed, in one line: April impressions IS the label, wearing a
column name. The model did not get smarter - it got handed the answer. Nothing in the
dataframe warned me; the column sorted alphabetically between two legitimate ones and
had a perfectly reasonable name. The only defence is the window rule in section 1,
enforced by the assert above rather than remembered.
""")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**The named limitation of my slice: left-censoring at the window edge.**

A content item that was registered to the platform partway through March accrues daily rows only
from its registration day. Its `impressions_march` is therefore a partial-month total compared
against a full-month April — so the *ratio* my label is built from is biased downward for those
items, and they will look like recoveries when they are only latecomers finally being measured
for a whole month. The next cell counts them; if the share is small I filter them out and say so,
and if it is large the eligibility rule needs rewriting before any modelling week.

**Three more the data can never answer, whatever I do to it:**

- **Why.** An observational panel. A page that fell may have been outranked, seasonal,
  cannibalised by a sibling page, or hit by an algorithm update — the columns cannot distinguish
  these, and no amount of feature engineering will make them.
- **Whether refreshing helps.** There is no randomised holdout of refreshed vs. non-refreshed
  pages anywhere in this release. Causal language is off the table permanently, not just for now.
- **What happened inside the shop before each client's `ga4_data_start`.** The unbalanced panel
  is a property of when each client switched their export on; the missing months are not
  recoverable, because exports have no time machine.

**One more that belongs to my choice, not to the data:** April 2026 is one month. A page's
decline against one 30-day window may be seasonality rather than decay. The honest version of
this experiment repeats it across several decision dates and checks the label holds — that is a
modelling-weeks job, and I am flagging it now so the first number I report is read as *one month,
one client-held-out split*, not as a stable estimate.

In [ ]:
# --- Quantify the named limitation -------------------------------------------
late = frame["first_seen_march"] > np.datetime64("2026-03-01")
n_late = int(late.sum())

print(f"Content items first seen after 1 March : {n_late:,} of {len(frame):,} "
      f"({n_late / len(frame):.1%})")

if n_late:
    print(f"\nTheir label rate  : {frame.loc[late, 'will_decline_next_30d'].mean():.3f}")
    print(f"Everyone else     : {frame.loc[~late, 'will_decline_next_30d'].mean():.3f}")
    print("\nIf those two rates differ noticeably, the gap is an artefact of the window,")
    print("not a finding about content - partial March totals inflate the April ratio.")

frame_clean = frame.loc[~late].copy()
print(f"\nAfter dropping partial-month items: {len(frame_clean):,} rows "
      f"(base rate {frame_clean['will_decline_next_30d'].mean():.3f})")
print("Carried into the modelling weeks as the eligibility rule, written down here so it")
print("is a decision on the record rather than a filter someone finds in the code later.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] Five contract answers in plain words: grain · tables · window · label · one deliberate exclusion
- [ ] Exactly three verification queries, outputs visible, availability checked with `IS TRUE`
- [ ] Five features (four if this release ships no position column), each with an "available when?" line
- [ ] The deliberate leak was added, scored, and **deleted** — the kept number is the honest one
- [ ] One named limitation of my slice, quantified rather than asserted
- [ ] The `_sample` table was never used for label logic (it is the sealed final month)
- [ ] No client names, URLs, or private queries anywhere; the token lives in a Secret, not a cell
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.